# vit-backend-adaptive: setup notebook

Cells 1 to 6 of the study: configuration, environment probe, NVML power-counter
characterization, datasets, fine-tuning, and backend builders with a smoke test.
Later stages run as scripts (`cell7_accuracy.py` onward, see the README).

Outputs are cleared in this public copy. The measured values reported in the
manuscript (counter update interval, energy noise floor) are stored in
`output/noise_floor_<GPU>.json`.

## Cell 1: central configuration

Detects the execution environment (local WSL2 or Kaggle), fixes the seeds,
declares the backend, model, dataset, batch and resolution grids, and creates
the project folders.

In [ ]:
# === Cell 1: central configuration ===
import os, sys, json, platform, random
from pathlib import Path
from dataclasses import dataclass, field, asdict
import numpy as np
import torch

# ---------- environment detection ----------
IS_KAGGLE = Path("/kaggle/working").exists()
if IS_KAGGLE:
    ROOT = Path("/kaggle/working/vit_backend_adaptive")
else:
    ROOT = Path.home() / "projects" / "vit_backend_adaptive"

DIRS = {k: ROOT / k for k in ["fig", "dataset", "output", "model", "logs"]}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

# ---------- seeds ----------
SEEDS = [42, 1234, 2024]
def set_seed(s: int):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)
set_seed(SEEDS[0])

# ---------- hardware id ----------
def hardware_id() -> str:
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        return name.replace("NVIDIA ", "").replace(" ", "_")
    return "CPU_" + platform.processor().replace(" ", "_")
HW_ID = hardware_id()

# ---------- study grid ----------
@dataclass
class Config:
    backends: list = field(default_factory=lambda: [
        "eager", "torchscript", "inductor_default", "inductor_maxautotune",
        "ort_cuda_fp32", "ort_cuda_fp16", "ort_trt",
    ])
    backends_cpu: list = field(default_factory=lambda: [
        "eager", "torchscript", "inductor_default", "ort_cpu_fp32",
    ])
    models: list = field(default_factory=lambda: [
        "vit_tiny_patch16_224", "vit_small_patch16_224",
        "vit_base_patch16_224", "deit_small_patch16_224",
    ])
    datasets: list = field(default_factory=lambda: ["cifar10", "cifar100", "imagenette"])
    batch_sizes: list = field(default_factory=lambda: [1, 8, 32])
    resolutions: list = field(default_factory=lambda: [224, 384])
    fps_targets: list = field(default_factory=lambda: [30, 60, 120])
    n_warmup: int = 50
    n_timed: int = 200
    power_sample_ms: int = 10
    seeds: list = field(default_factory=lambda: SEEDS)
    hw_id: str = HW_ID
    root: str = str(ROOT)

CFG = Config()
if not torch.cuda.is_available():
    CFG.backends = CFG.backends_cpu
    CFG.batch_sizes = [1, 8]

# ---------- persist ----------
cfg_path = DIRS["output"] / f"config_{HW_ID}.json"
cfg_path.write_text(json.dumps(asdict(CFG), indent=2))

print(f"env      : {'kaggle' if IS_KAGGLE else 'local'}")
print(f"hardware : {HW_ID}")
print(f"torch    : {torch.__version__}  cuda={torch.version.cuda}")
print(f"cuda ok  : {torch.cuda.is_available()}")
print(f"root     : {ROOT}")
print(f"backends : {CFG.backends}")
print(f"config   : {cfg_path}")

In [ ]:
# --- library paths for ORT CUDA/TRT EPs (WSL2 + pip-installed CUDA libs) ---
import site, os
sp = site.getsitepackages()[0]
lib_dirs = [f"{sp}/tensorrt_libs", f"{sp}/nvidia/cublas/lib", f"{sp}/nvidia/cudnn/lib",
            f"{sp}/nvidia/cuda_runtime/lib", f"{sp}/nvidia/cufft/lib"]
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
try:
    import ctypes
    for d in lib_dirs:
        if os.path.isdir(d):
            for f_ in sorted(os.listdir(d)):
                if ".so" in f_:
                    try: ctypes.CDLL(os.path.join(d, f_), mode=ctypes.RTLD_GLOBAL)
                    except OSError: pass
except Exception as e:
    print("lib preload note:", e)

In [ ]:
# --- C compiler for torch.compile/Inductor (conda-forge gcc, apt unavailable) ---
import glob
cc_candidates = glob.glob(f"{sys.prefix}/bin/*-conda-linux-gnu-gcc") + [f"{sys.prefix}/bin/gcc"]
cc = next((c for c in cc_candidates if os.path.exists(c)), None)
if cc:
    os.environ["CC"] = cc
    os.environ["CXX"] = cc.replace("gcc", "g++")
    print(f"CC       : {cc}")
else:
    print("CC       : NOT FOUND (Inductor will fail)")

## Cell 2: environment probe

Records package versions, lists the available ONNX Runtime execution
providers (TensorRT availability), reads NVML power, clock and memory values,
and tests whether GPU clock locking is permitted.

In [ ]:
# === Cell 2: environment probe ===
import sys, json, subprocess, time, importlib.metadata as md
import torch, onnxruntime as ort
import pynvml

probe = {}

# ---------- versions ----------
pkgs = ["torch", "torchvision", "timm", "onnx", "onnxruntime-gpu",
        "tensorrt", "numpy", "pandas", "scipy", "nvidia-ml-py"]
probe["versions"] = {}
for p in pkgs:
    try:
        probe["versions"][p] = md.version(p)
    except md.PackageNotFoundError:
        probe["versions"][p] = "NOT INSTALLED"
probe["versions"]["cuda_torch"] = torch.version.cuda
probe["versions"]["cudnn"] = str(torch.backends.cudnn.version())
probe["versions"]["python"] = sys.version.split()[0]

# ---------- ORT execution providers ----------
probe["ort_available_eps"] = ort.get_available_providers()
probe["trt_ep_available"] = "TensorrtExecutionProvider" in probe["ort_available_eps"]

# ---------- NVML probe ----------
pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
name = pynvml.nvmlDeviceGetName(h)
probe["gpu_name"] = name.decode() if isinstance(name, bytes) else name
probe["driver"] = pynvml.nvmlSystemGetDriverVersion()

# power: 20 quick samples to confirm the counter responds
samples = []
for _ in range(20):
    samples.append(pynvml.nvmlDeviceGetPowerUsage(h) / 1000.0)  # W
    time.sleep(0.01)
probe["power_W_min_max"] = [min(samples), max(samples)]
probe["power_limit_W"] = pynvml.nvmlDeviceGetEnforcedPowerLimit(h) / 1000.0

# clocks
probe["clock_sm_MHz"] = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_SM)
probe["clock_mem_MHz"] = pynvml.nvmlDeviceGetClockInfo(h, pynvml.NVML_CLOCK_MEM)
probe["temp_C"] = pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)

# per-process memory attribution (not available under WSL2)
try:
    procs = pynvml.nvmlDeviceGetComputeRunningProcesses(h)
    probe["nvml_process_memory"] = [
        {"pid": p.pid, "MB": (p.usedGpuMemory / 1024**2) if p.usedGpuMemory else None}
        for p in procs
    ]
    probe["procmem_ok"] = any(p["MB"] for p in probe["nvml_process_memory"])
except pynvml.NVMLError as e:
    probe["nvml_process_memory"] = f"ERROR: {e}"
    probe["procmem_ok"] = False

# total memory fallback (always works)
mi = pynvml.nvmlDeviceGetMemoryInfo(h)
probe["mem_total_used_MB"] = [mi.total / 1024**2, mi.used / 1024**2]

# ---------- clock locking attempt ----------
def try_cmd(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return {"cmd": " ".join(cmd), "rc": r.returncode,
            "out": (r.stdout + r.stderr).strip()[:200]}
probe["clock_lock"] = try_cmd(["nvidia-smi", "-lgc", str(probe["clock_sm_MHz"])])
if probe["clock_lock"]["rc"] == 0:
    probe["clock_unlock"] = try_cmd(["nvidia-smi", "-rgc"])  # release immediately

pynvml.nvmlShutdown()

# ---------- persist + report ----------
out = DIRS["output"] / f"env_probe_{HW_ID}.json"
out.write_text(json.dumps(probe, indent=2))

print(json.dumps(probe["versions"], indent=2))
print("ORT EPs        :", probe["ort_available_eps"])
print("TRT EP         :", probe["trt_ep_available"])
print("power W range  :", probe["power_W_min_max"], "limit:", probe["power_limit_W"])
print("SM/mem clocks  :", probe["clock_sm_MHz"], "/", probe["clock_mem_MHz"], "MHz,",
      probe["temp_C"], "C")
print("proc-mem OK    :", probe["procmem_ok"], probe["nvml_process_memory"])
print("clock lock     :", "rc=", probe["clock_lock"]["rc"], "|", probe["clock_lock"]["out"][:120])
print("saved          :", out)

## Cell 3: NVML counter characterization and energy noise floor

Samples the NVML power counter every 2 ms for 3 s at idle and under a constant
synthetic load to find its true update interval, then repeats a fixed
reference workload ten times to obtain the 95% bootstrap half-width of the
energy estimate. Also records the idle device-memory baseline.

In [ ]:
# === Cell 3: NVML counter characterization + noise floor ===
import time, threading, json
import numpy as np
import torch
import pynvml

pynvml.nvmlInit()
h = pynvml.nvmlDeviceGetHandleByIndex(0)
res = {}

# ---------- (a) counter update interval: sample fast, count distinct values ----------
def counter_interval(duration_s=3.0, period_ms=2):
    t0 = time.perf_counter()
    ts, ps = [], []
    while time.perf_counter() - t0 < duration_s:
        ts.append(time.perf_counter())
        ps.append(pynvml.nvmlDeviceGetPowerUsage(h))
        time.sleep(period_ms / 1000)
    ps = np.array(ps); ts = np.array(ts)
    changes = ts[1:][np.diff(ps) != 0]
    intervals = np.diff(changes) * 1000 if len(changes) > 2 else np.array([np.nan])
    return {"n_samples": len(ps), "n_changes": int((np.diff(ps) != 0).sum()),
            "median_update_ms": float(np.nanmedian(intervals)),
            "power_W_range": [float(ps.min()/1000), float(ps.max()/1000)]}

res["idle_counter"] = counter_interval()

# constant synthetic load in background
stop = threading.Event()
def gpu_load():
    a = torch.randn(4096, 4096, device="cuda")
    while not stop.is_set():
        a = a @ a; a = a / a.norm()
    torch.cuda.synchronize()

th = threading.Thread(target=gpu_load); th.start()
time.sleep(2.0)  # ramp
res["load_counter"] = counter_interval()
stop.set(); th.join(); torch.cuda.synchronize(); torch.cuda.empty_cache()
time.sleep(2.0)

# ---------- (b) idle memory baseline ----------
mi = pynvml.nvmlDeviceGetMemoryInfo(h)
res["idle_mem_MB"] = float(mi.used / 1024**2)

# ---------- (c) energy noise floor: 10 identical reference measurements ----------
# reference workload: fixed matmul chain ~ a few hundred ms, akin to a timed window
def energy_of_reference():
    samples = []
    stop_p = threading.Event()
    def sampler():
        while not stop_p.is_set():
            samples.append((time.perf_counter(), pynvml.nvmlDeviceGetPowerUsage(h)/1000))
            time.sleep(0.01)
    a = torch.randn(2048, 2048, device="cuda")
    torch.cuda.synchronize()
    sp = threading.Thread(target=sampler); sp.start()
    t0 = time.perf_counter()
    for _ in range(200):
        a = a @ a; a = a / a.norm()
    torch.cuda.synchronize()
    t1 = time.perf_counter()
    stop_p.set(); sp.join()
    t = np.array([s[0] for s in samples]); p = np.array([s[1] for s in samples])
    mask = (t >= t0) & (t <= t1)
    if mask.sum() < 2: return np.nan, (t1 - t0)
    E = np.trapezoid(p[mask], t[mask])          # joules over window
    return E, (t1 - t0)

E_list, T_list = [], []
for i in range(10):
    E, T = energy_of_reference()
    E_list.append(E); T_list.append(T)
    time.sleep(1.0)

E_arr = np.array(E_list)
boot = np.random.default_rng(42).choice(E_arr, (5000, len(E_arr))).mean(axis=1)
ci = np.percentile(boot, [2.5, 97.5])
res["ref_energy_J"] = {"mean": float(E_arr.mean()), "std": float(E_arr.std(ddof=1)),
                       "ci95": [float(ci[0]), float(ci[1])],
                       "ci_halfwidth_pct": float((ci[1]-ci[0])/2/E_arr.mean()*100),
                       "window_s_mean": float(np.mean(T_list))}

pynvml.nvmlShutdown()
out = DIRS["output"] / f"noise_floor_{HW_ID}.json"
out.write_text(json.dumps(res, indent=2))

print("idle counter :", res["idle_counter"])
print("load counter :", res["load_counter"])
print("idle mem MB  :", round(res["idle_mem_MB"], 1))
print("ref energy   : mean %.2f J, std %.3f, CI half-width %.2f%%" % (
    res["ref_energy_J"]["mean"], res["ref_energy_J"]["std"],
    res["ref_energy_J"]["ci_halfwidth_pct"]))
print("saved        :", out)

## Cell 4: datasets

Downloads and prepares CIFAR-10, CIFAR-100 and Imagenette under `dataset/`,
with train and test transforms per resolution (224 or 384 px) and
per-dataset normalization. Prints split sizes as a sanity check.

In [ ]:
# === Cell 4: datasets ===
import tarfile, urllib.request
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader

DATA = DIRS["dataset"]

# per-dataset normalization (train-split statistics, standard values)
NORM = {
    "cifar10":    ([0.4914, 0.4822, 0.4465], [0.2470, 0.2435, 0.2616]),
    "cifar100":   ([0.5071, 0.4865, 0.4409], [0.2673, 0.2564, 0.2762]),
    "imagenette": ([0.485, 0.456, 0.406],   [0.229, 0.224, 0.225]),  # ImageNet stats
}

def transforms_for(ds: str, res: int, train: bool):
    mean, std = NORM[ds]
    if train:
        return T.Compose([
            T.RandomResizedCrop(res, scale=(0.8, 1.0)),
            T.RandomHorizontalFlip(),
            T.ToTensor(), T.Normalize(mean, std),
        ])
    return T.Compose([
        T.Resize(int(res * 256 / 224)), T.CenterCrop(res),
        T.ToTensor(), T.Normalize(mean, std),
    ])

def get_dataset(ds: str, res: int, train: bool):
    tf = transforms_for(ds, res, train)
    if ds == "cifar10":
        return torchvision.datasets.CIFAR10(DATA, train=train, download=True, transform=tf)
    if ds == "cifar100":
        return torchvision.datasets.CIFAR100(DATA, train=train, download=True, transform=tf)
    if ds == "imagenette":
        split = "train" if train else "val"
        root = DATA / "imagenette2"
        if not root.exists():
            url = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2.tgz"
            tgz = DATA / "imagenette2.tgz"
            print("downloading imagenette (~1.5 GB)...")
            urllib.request.urlretrieve(url, tgz)
            with tarfile.open(tgz) as t_:
                t_.extractall(DATA)
            tgz.unlink()
        return torchvision.datasets.ImageFolder(root / split, transform=tf)
    raise ValueError(ds)

N_CLASSES = {"cifar10": 10, "cifar100": 100, "imagenette": 10}

# sanity check at 224
for ds in CFG.datasets:
    tr = get_dataset(ds, 224, True); te = get_dataset(ds, 224, False)
    x, y = tr[0]
    print(f"{ds:11s} train={len(tr):6d} test={len(te):6d} classes={N_CLASSES[ds]:3d} sample={tuple(x.shape)}")

## Cell 5: fine-tuning (12 checkpoints)

Fine-tunes each of the four models on each of the three datasets at 224 px
for 10 epochs with AdamW and a cosine schedule. Pairs with an existing
checkpoint are skipped. Outputs: `model/{model}_{dataset}_best.pth` and
`output/finetune_log.csv`. All backend measurements use these checkpoints.

In [ ]:
# === Cell 5: fine-tuning, resume-safe ===
import csv, time
import torch, torch.nn as nn
import timm
from torch.utils.data import DataLoader

EPOCHS = 10
LR = 3e-4
WD = 1e-4
BS_TRAIN = 128
BS_EVAL = 256

log_path = DIRS["output"] / "finetune_log.csv"
if not log_path.exists():
    log_path.write_text("model,dataset,epoch,train_loss,train_acc,test_acc,seconds\n")

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            with torch.autocast("cuda"):
                out = model(x)
            correct += (out.argmax(1) == y).sum().item(); total += y.numel()
    return 100 * correct / total

def finetune(model_name: str, ds: str):
    ckpt = DIRS["model"] / f"{model_name}_{ds}_best.pth"
    if ckpt.exists():
        print(f"skip {model_name}/{ds} (checkpoint exists)")
        return
    set_seed(SEEDS[0])
    tr = get_dataset(ds, 224, True); te = get_dataset(ds, 224, False)
    trl = DataLoader(tr, batch_size=BS_TRAIN, shuffle=True, num_workers=8,
                     pin_memory=True, persistent_workers=True)
    tel = DataLoader(te, batch_size=BS_EVAL, shuffle=False, num_workers=8,
                     pin_memory=True, persistent_workers=True)
    model = timm.create_model(model_name, pretrained=True,
                              num_classes=N_CLASSES[ds]).cuda()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    crit = nn.CrossEntropyLoss()
    best = 0.0
    for ep in range(1, EPOCHS + 1):
        t0 = time.time(); model.train()
        run_loss = correct = total = 0
        for x, y in trl:
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            run_loss += loss.item() * y.numel()
            correct += (out.argmax(1) == y).sum().item(); total += y.numel()
        sched.step()
        tr_acc = 100 * correct / total
        te_acc = evaluate(model, tel)
        secs = time.time() - t0
        with open(log_path, "a") as f:
            csv.writer(f).writerow([model_name, ds, ep,
                f"{run_loss/total:.4f}", f"{tr_acc:.2f}", f"{te_acc:.2f}", f"{secs:.0f}"])
        if te_acc > best:
            best = te_acc
            torch.save(model.state_dict(), ckpt)
        print(f"{model_name}/{ds} ep{ep:02d} loss={run_loss/total:.4f} "
              f"train={tr_acc:.2f} test={te_acc:.2f} ({secs:.0f}s)")
    print(f"DONE {model_name}/{ds} best={best:.2f} -> {ckpt.name}")
    del model; torch.cuda.empty_cache()

for m in CFG.models:
    for ds in CFG.datasets:
        finetune(m, ds)
print("All 12 checkpoints ready.")

## Cell 6: backend builders and setup-cost measurement

One builder per backend behind a common interface,
`build_backend(name, model_name, dataset, batch, res)`, returning an inference
callable and the setup time. Each setup stage (ONNX export, compilation or
engine build, session creation, first inference) is logged to
`output/setup_costs_<GPU>.csv`; these records are the raw data for the
switching costs. TensorRT engines are cached by ONNX Runtime per shape.
A smoke test runs every backend on ViT-Tiny / CIFAR-10, batch 1, 224 px and
reports the maximum logit difference from eager execution.

In [ ]:
# === Cell 6: backend builders + setup cost ===
import time, json, csv
import numpy as np
import torch
import timm
import onnxruntime as ort

TRT_CACHE = DIRS["model"] / "trt_cache"; TRT_CACHE.mkdir(exist_ok=True)
SETUP_CSV = DIRS["output"] / f"setup_costs_{HW_ID}.csv"
if not SETUP_CSV.exists():
    SETUP_CSV.write_text("backend,model,dataset,batch,res,stage,seconds\n")

def _log_setup(backend, model_name, ds, batch, res, stage, secs):
    with open(SETUP_CSV, "a") as f:
        csv.writer(f).writerow([backend, model_name, ds, batch, res, stage, f"{secs:.3f}"])

def load_finetuned(model_name, ds):
    m = timm.create_model(model_name, pretrained=False, num_classes=N_CLASSES[ds])
    sd = torch.load(DIRS["model"] / f"{model_name}_{ds}_best.pth",
                    map_location="cpu", weights_only=True)
    m.load_state_dict(sd); m.eval()
    return m

def export_onnx(model_name, ds, res):
    """One ONNX file per (model, dataset, res); dynamic batch axis."""
    path = DIRS["model"] / f"{model_name}_{ds}_{res}.onnx"
    if path.exists():
        return path, 0.0
    t0 = time.perf_counter()
    m = load_finetuned(model_name, ds)
    dummy = torch.randn(1, 3, res, res)
    torch.onnx.export(m, dummy, path,
                      input_names=["input"], output_names=["logits"],
                      dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
                      opset_version=17)
    return path, time.perf_counter() - t0

def build_backend(name, model_name, ds, batch, res):
    """Returns (infer_fn, setup_total_s). infer_fn: gpu tensor -> gpu logits (torch)."""
    t_start = time.perf_counter()
    x_warm = torch.randn(batch, 3, res, res, device="cuda")

    if name in ("eager", "torchscript", "inductor_default", "inductor_maxautotune"):
        m = load_finetuned(model_name, ds).cuda()
        if name == "torchscript":
            t0 = time.perf_counter()
            m = torch.jit.script(m)
            _log_setup(name, model_name, ds, batch, res, "compile", time.perf_counter() - t0)
        elif name.startswith("inductor"):
            mode = "max-autotune" if name.endswith("maxautotune") else "default"
            t0 = time.perf_counter()
            m = torch.compile(m, mode=mode, dynamic=False)
            _log_setup(name, model_name, ds, batch, res, "compile_call", time.perf_counter() - t0)
        @torch.no_grad()
        def infer(x): return m(x)

    elif name.startswith("ort_"):
        onnx_path, t_exp = export_onnx(model_name, ds, res)
        if t_exp > 0: _log_setup(name, model_name, ds, batch, res, "onnx_export", t_exp)
        so = ort.SessionOptions()
        if name == "ort_trt":
            providers = [("TensorrtExecutionProvider", {
                "trt_engine_cache_enable": True,
                "trt_engine_cache_path": str(TRT_CACHE),
                "trt_fp16_enable": True,
                "trt_timing_cache_enable": True,
            }), "CUDAExecutionProvider", "CPUExecutionProvider"]
        elif name == "ort_cuda_fp16":
            # The FP16 graphs used in the study are produced by cell7_accuracy.py
            # and cell8_grid.py with the ONNX Runtime transformer optimizer
            # (per-model attention parameters). This conversion is only a
            # smoke-test fallback when no FP16 graph exists yet.
            fp16_path = onnx_path.with_name(onnx_path.stem + "_fp16.onnx")
            if not fp16_path.exists():
                t0 = time.perf_counter()
                from onnxconverter_common import float16
                import onnx as onnx_lib
                mdl = onnx_lib.load(str(onnx_path))
                mdl16 = float16.convert_float_to_float16(mdl, keep_io_types=True)
                onnx_lib.save(mdl16, str(fp16_path))
                _log_setup(name, model_name, ds, batch, res, "fp16_convert", time.perf_counter() - t0)
            onnx_path = fp16_path
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        elif name == "ort_cuda_fp32":
            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
        else:  # ort_cpu_fp32
            providers = ["CPUExecutionProvider"]
        t0 = time.perf_counter()
        sess = ort.InferenceSession(str(onnx_path), so, providers=providers)
        _log_setup(name, model_name, ds, batch, res, "session_create", time.perf_counter() - t0)
        active = sess.get_providers()[0]
        if name == "ort_trt" and active != "TensorrtExecutionProvider":
            raise RuntimeError(f"TRT EP did not load; active={active}")
        def infer(x):
            out = sess.run(["logits"], {"input": x.cpu().numpy()})[0]
            return torch.from_numpy(out).cuda()
    else:
        raise ValueError(name)

    # first warm inference (includes lazy compilation / engine build)
    t0 = time.perf_counter()
    with torch.no_grad():
        _ = infer(x_warm)
    torch.cuda.synchronize()
    _log_setup(name, model_name, ds, batch, res, "first_infer", time.perf_counter() - t0)

    total = time.perf_counter() - t_start
    _log_setup(name, model_name, ds, batch, res, "TOTAL", total)
    return infer, total

# ---------- smoke test: vit_tiny / cifar10 / batch 1 / 224 ----------
pip_note = None
try:
    import onnxconverter_common  # needed for ort_cuda_fp16
except ImportError:
    pip_note = "onnxconverter-common is required: pip install onnxconverter-common, then rerun this cell"

if pip_note:
    print(pip_note)
else:
    set_seed(SEEDS[0])
    x_test = torch.randn(1, 3, 224, 224, device="cuda")
    ref = None
    results = {}
    for be in CFG.backends:
        try:
            infer, total = build_backend(be, "vit_tiny_patch16_224", "cifar10", 1, 224)
            with torch.no_grad():
                out = infer(x_test).float().cpu()
            if ref is None and be == "eager":
                ref = out
            diff = (out - ref).abs().max().item() if ref is not None else 0.0
            results[be] = {"setup_s": round(total, 2), "max_abs_diff_vs_eager": round(diff, 5)}
            print(f"{be:22s} setup={total:7.2f}s  maxdiff={diff:.5f}")
        except Exception as e:
            results[be] = {"error": str(e)[:200]}
            print(f"{be:22s} FAILED: {str(e)[:120]}")
    (DIRS["output"] / f"smoke_{HW_ID}.json").write_text(json.dumps(results, indent=2))